# 06 — Multi-modal Data Augmentation

References:
- [AgaMiko/data-augmentation-review](https://github.com/AgaMiko/data-augmentation-review)
- [facebookresearch/AugLy](https://github.com/facebookresearch/AugLy)
- [tensorflow.org/tutorials/images/data_augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation)

**Runtime → T4 GPU**

| Modality | Library |
|---|---|
| Image | TF / albumentations / keras_cv |
| Text | nlpaug |
| Time Series | tsaug |
| Tabular | SMOTE / noise / feature dropout |
| Speech | audiomentations |
| Video | AugLy (video) |
| Document Images | albumentations / AugLy |

In [ ]:
# Install all libraries (run once, ~3 min)
!pip install -q \
    augly \
    nlpaug \
    albumentations \
    audiomentations \
    tsaug \
    imbalanced-learn \
    transformers \
    librosa \
    opencv-python-headless

# Suppress noisy warnings
import warnings; warnings.filterwarnings('ignore')
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
print('All libraries installed.')

---
## PART A — Image Augmentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import cv2

# ── Load CIFAR-10 as uint8 for albumentations ────────────────────────────────
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test  = y_test.flatten()
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
print('Image dataset ready:', x_train_raw.shape)

In [ ]:
# ── A1: TensorFlow / Keras augmentation ─────────────────────────────────────
tf_aug = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1),
], name='tf_augmentation')

sample_f = x_train_raw[:8].astype('float32') / 255.0
aug_f    = tf_aug(sample_f, training=True).numpy()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0,i].imshow(sample_f[i]); axes[0,i].axis('off')
    axes[1,i].imshow(aug_f[i]);    axes[1,i].axis('off')
axes[0,0].set_ylabel('Original', fontsize=9)
axes[1,0].set_ylabel('TF Aug',   fontsize=9)
plt.suptitle('A1: TensorFlow/Keras Image Augmentation', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── A2: Albumentations ───────────────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

albu_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=20, p=0.6),
    A.OneOf([
        A.GaussNoise(var_limit=(5, 30)),
        A.GaussianBlur(blur_limit=3),
        A.MotionBlur(blur_limit=3),
    ], p=0.4),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10),
        A.CLAHE(clip_limit=2),
    ], p=0.5),
    A.CoarseDropout(max_holes=4, max_height=8, max_width=8, p=0.3),  # Cutout
    A.ElasticTransform(alpha=1, sigma=5, p=0.2),
    A.GridDistortion(num_steps=3, distort_limit=0.1, p=0.2),
])

sample_u8 = x_train_raw[:8]  # uint8
albu_out  = np.array([albu_pipeline(image=img)['image'] for img in sample_u8])

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0,i].imshow(sample_u8[i]);  axes[0,i].axis('off')
    axes[1,i].imshow(albu_out[i]);   axes[1,i].axis('off')
axes[0,0].set_ylabel('Original',     fontsize=9)
axes[1,0].set_ylabel('Albumentations', fontsize=9)
plt.suptitle('A2: Albumentations Pipeline', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── A3: Train with/without albumentations — A/B test ─────────────────────────
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

class CIFARAug(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(image=img)['image']
        return torch.tensor(img, dtype=torch.float32).permute(2,0,1) / 255.0, self.labels[idx]

import torchvision.transforms as T
torch_norm = T.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))

no_aug_ds  = CIFARAug(x_train_raw, y_train, transform=None)
albu_ds    = CIFARAug(x_train_raw, y_train, transform=albu_pipeline)
test_ds_pt = CIFARAug(x_test_raw,  y_test,  transform=None)

print('PyTorch datasets ready for A/B test.')

---
## PART B — Text Augmentation (nlpaug)

In [ ]:
import nlpaug.augmenter.word  as naw
import nlpaug.augmenter.char  as nac
import nlpaug.augmenter.sentence as nas

# Sample texts
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Deep learning models require large amounts of training data.",
    "Data augmentation helps prevent overfitting in neural networks.",
]

print('=== B1: Synonym Replacement (WordNet) ===')
aug_syn = naw.SynonymAug(aug_src='wordnet', aug_p=0.3)
for t in texts:
    print(f'  Original : {t}')
    print(f'  Augmented: {aug_syn.augment(t)[0]}\n')

In [ ]:
print('=== B2: Random Word Deletion ===')
aug_del = naw.RandomWordAug(action='delete', aug_p=0.2)
for t in texts:
    print(f'  Original : {t}')
    print(f'  Augmented: {aug_del.augment(t)[0]}\n')

In [ ]:
print('=== B3: Random Word Swap ===')
aug_swap = naw.RandomWordAug(action='swap', aug_p=0.2)
for t in texts:
    print(f'  Original : {t}')
    print(f'  Augmented: {aug_swap.augment(t)[0]}\n')

In [ ]:
print('=== B4: Character-level Augmentation (OCR errors, keyboard typos) ===')
aug_ocr = nac.OcrAug(aug_p=0.2)
aug_kb  = nac.KeyboardAug(aug_p=0.1)

for t in texts[:2]:
    print(f'  Original: {t}')
    print(f'  OCR err : {aug_ocr.augment(t)[0]}')
    print(f'  KB typo : {aug_kb.augment(t)[0]}\n')

In [ ]:
print('=== B5: Back-Translation simulation (contextual word insertion) ===')
aug_ctx = naw.ContextualWordEmbsAug(
    model_path='bert-base-uncased',
    action='insert',
    aug_p=0.15,
    device='cpu'
)
for t in texts[:2]:
    print(f'  Original : {t}')
    print(f'  Augmented: {aug_ctx.augment(t)[0]}\n')

In [ ]:
# ── B6: Text augmentation for classification A/B test ────────────────────────
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

categories = ['sci.space','comp.graphics','rec.sport.baseball','talk.politics.guns']
news = fetch_20newsgroups(subset='train', categories=categories, remove=('headers','footers','quotes'))
news_test = fetch_20newsgroups(subset='test', categories=categories, remove=('headers','footers','quotes'))

# Baseline: no augmentation
vec  = TfidfVectorizer(max_features=5000)
X_tr = vec.fit_transform(news.data)
X_te = vec.transform(news_test.data)
cls  = LogisticRegression(max_iter=1000).fit(X_tr, news.target)
acc_base = accuracy_score(news_test.target, cls.predict(X_te))
print(f'Baseline (no aug) accuracy: {acc_base:.4f}')

# Augmented: add synonym-replaced copies
print('Augmenting training texts (synonym replacement)...')
aug_texts  = []
aug_labels = []
for txt, lbl in zip(news.data[:500], news.target[:500]):  # limit for speed
    try:
        augmented = aug_syn.augment(txt)[0]
        aug_texts.append(augmented)
        aug_labels.append(lbl)
    except Exception:
        pass

all_texts  = list(news.data) + aug_texts
all_labels = list(news.target) + aug_labels
X_tr_aug  = vec.transform(all_texts)
cls_aug   = LogisticRegression(max_iter=1000).fit(X_tr_aug, all_labels)
acc_aug   = accuracy_score(news_test.target, cls_aug.predict(X_te))
print(f'With text augmentation accuracy: {acc_aug:.4f}')

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(['No Augmentation','Synonym Aug'], [acc_base, acc_aug],
       color=['#3498db','#e74c3c'], edgecolor='k', width=0.4)
ax.set_ylim(0.7, 1.0); ax.set_title('Text A/B Test: 20 Newsgroups Classification')
for i, v in enumerate([acc_base, acc_aug]):
    ax.text(i, v+0.003, f'{v:.3f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## PART C — Time Series Augmentation (tsaug)

In [ ]:
import tsaug
from tsaug import TimeWarp, Crop, Quantize, Drift, Reverse, AddNoise, Pool

# Simulate ECG-like time series
np.random.seed(42)
t   = np.linspace(0, 4*np.pi, 200)
ts  = np.sin(t) + 0.1*np.sin(5*t) + 0.05*np.random.randn(200)  # shape (200,)
TS  = ts[np.newaxis, :, np.newaxis]  # tsaug expects (n_samples, seq_len, n_channels)

augs = {
    'Original'    : lambda x: x,
    'TimeWarp'    : lambda x: TimeWarp(n_speed_change=3, max_speed_ratio=3.0).augment(x),
    'AddNoise'    : lambda x: AddNoise(scale=0.05).augment(x),
    'Drift'       : lambda x: Drift(max_drift=0.3, n_drift_points=5).augment(x),
    'Quantize'    : lambda x: Quantize(n_levels=20).augment(x),
    'Reverse'     : lambda x: Reverse().augment(x),
    'Pool'        : lambda x: Pool(size=3).augment(x),
    'Crop+Pad'    : lambda x: Crop(size=150).augment(x),
}

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, (name, aug_fn) in zip(axes.flat, augs.items()):
    try:
        ts_aug = aug_fn(TS.copy())[0, :, 0]
        ax.plot(ts_aug, color='steelblue', lw=1.2)
    except Exception:
        ax.plot(ts, color='gray', lw=1.2)
    ax.set_title(name); ax.grid(True, alpha=0.3); ax.set_xticks([])
plt.suptitle('C: Time Series Augmentation (tsaug)', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── C2: Time series classification A/B test ───────────────────────────────────
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Create synthetic multivariate time-series classification dataset
np.random.seed(0)
n_samples, seq_len = 500, 100

def make_ts_dataset(n=500, seq=100):
    X, y = [], []
    for cls in range(3):
        for _ in range(n//3):
            t   = np.linspace(0, 2*np.pi, seq)
            sig = np.sin((cls+1)*t) + 0.15*np.random.randn(seq)
            X.append(sig); y.append(cls)
    return np.array(X), np.array(y)

X_ts, y_ts = make_ts_dataset()
X_tr, X_te, y_tr, y_te = train_test_split(X_ts, y_ts, test_size=0.2, random_state=42)

# Flatten for RF
clf_base = RandomForestClassifier(n_estimators=50, random_state=42)
clf_base.fit(X_tr, y_tr)
acc_ts_base = accuracy_score(y_te, clf_base.predict(X_te))

# Augment: add noise + time-warp versions
TS_3d = X_tr[:, :, np.newaxis]   # (n, seq, 1)
noise_aug = AddNoise(scale=0.05)
warp_aug  = TimeWarp(n_speed_change=2, max_speed_ratio=2.0)

X_aug_list = [X_tr]
for aug_fn in [noise_aug, warp_aug]:
    try:
        aug_out = aug_fn.augment(TS_3d)[:, :, 0]
        X_aug_list.append(aug_out)
    except Exception as e:
        print(f'  Warning: {e}')

X_aug = np.vstack(X_aug_list)
y_aug = np.tile(y_tr, len(X_aug_list))

clf_aug = RandomForestClassifier(n_estimators=50, random_state=42)
clf_aug.fit(X_aug, y_aug)
acc_ts_aug = accuracy_score(y_te, clf_aug.predict(X_te))

print(f'Time Series — No Aug: {acc_ts_base:.4f} | With Aug: {acc_ts_aug:.4f}')
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(['No Augmentation','Noise+TimeWarp'], [acc_ts_base, acc_ts_aug],
       color=['#3498db','#2ecc71'], edgecolor='k', width=0.4)
ax.set_ylim(0.6, 1.0); ax.set_title('Time Series Classification A/B Test')
for i,v in enumerate([acc_ts_base, acc_ts_aug]):
    ax.text(i, v+0.003, f'{v:.3f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---
## PART D — Tabular Data Augmentation

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# ── D1: Load dataset ─────────────────────────────────────────────────────────
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Baseline
clf = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
acc_tab_base = accuracy_score(y_te, clf.predict(X_te_s))
print(f'Baseline accuracy: {acc_tab_base:.4f}')

# ── D2: Gaussian noise augmentation ─────────────────────────────────────────
X_noisy = X_tr_s + np.random.normal(0, 0.05, X_tr_s.shape)
X_aug_tab = np.vstack([X_tr_s, X_noisy])
y_aug_tab = np.concatenate([y_tr, y_tr])
clf_noise = LogisticRegression(max_iter=1000).fit(X_aug_tab, y_aug_tab)
acc_noise = accuracy_score(y_te, clf_noise.predict(X_te_s))
print(f'Gaussian noise aug accuracy: {acc_noise:.4f}')

# ── D3: SMOTE (Synthetic Minority Over-sampling) ─────────────────────────────
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_tr_s, y_tr)
clf_smote = LogisticRegression(max_iter=1000).fit(X_sm, y_sm)
acc_smote = accuracy_score(y_te, clf_smote.predict(X_te_s))
print(f'SMOTE accuracy: {acc_smote:.4f}')

# ── D4: Feature dropout (randomly zero-out features during training) ──────────
X_fdrop = X_tr_s.copy()
mask = np.random.rand(*X_fdrop.shape) < 0.1   # 10% dropout per feature
X_fdrop[mask] = 0
X_aug_fd = np.vstack([X_tr_s, X_fdrop])
y_aug_fd = np.concatenate([y_tr, y_tr])
clf_fd = LogisticRegression(max_iter=1000).fit(X_aug_fd, y_aug_fd)
acc_fd = accuracy_score(y_te, clf_fd.predict(X_te_s))
print(f'Feature dropout accuracy: {acc_fd:.4f}')

# Plot
names = ['Baseline','Gaussian Noise','SMOTE','Feature Dropout']
accs  = [acc_tab_base, acc_noise, acc_smote, acc_fd]
fig, ax = plt.subplots(figsize=(9,4))
bars = ax.bar(names, accs, color=['#e74c3c','#3498db','#2ecc71','#9b59b6'], edgecolor='k', width=0.5)
ax.set_ylim(0.88, 1.0); ax.set_title('Tabular Augmentation — Breast Cancer Classification')
for bar, v in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---
## PART E — Speech / Audio Augmentation (audiomentations)

In [ ]:
import audiomentations as am
import librosa
import librosa.display
from IPython.display import Audio

# Load sample audio (download a free wav)
!wget -q -O /tmp/speech.wav https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3 2>/dev/null || \
 python3 -c "
import numpy as np; import soundfile as sf
sr = 16000; t = np.linspace(0, 2, 2*sr)
signal = (0.5*np.sin(2*np.pi*440*t) + 0.3*np.sin(2*np.pi*880*t)).astype('float32')
sf.write('/tmp/speech.wav', signal, sr)"

try:
    audio, sr = librosa.load('/tmp/speech.wav', sr=16000, duration=3.0, mono=True)
except Exception:
    sr = 16000
    t  = np.linspace(0, 3, 3*sr)
    audio = (0.5*np.sin(2*np.pi*440*t) + 0.2*np.sin(2*np.pi*880*t)).astype('float32')

print(f'Audio shape: {audio.shape}, Sample rate: {sr}')

In [ ]:
# ── E1: Define audio augmentation pipeline ───────────────────────────────────
audio_aug_pipeline = am.Compose([
    am.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.8),
    am.TimeStretch(min_rate=0.8, max_rate=1.2, p=0.5),
    am.PitchShift(min_semitones=-3, max_semitones=3, p=0.5),
    am.Shift(min_fraction=-0.2, max_fraction=0.2, p=0.5),
    am.Reverse(p=0.2),
    am.Gain(min_gain_in_db=-6, max_gain_in_db=6, p=0.5),
])

# Apply augmentations
N_AUG = 4
fig, axes = plt.subplots(N_AUG+1, 1, figsize=(14, 2*(N_AUG+1)))

axes[0].plot(audio, lw=0.5, color='steelblue')
axes[0].set_title('Original Audio Waveform')
axes[0].set_ylabel('Amplitude')

for i in range(N_AUG):
    aug_audio = audio_aug_pipeline(samples=audio.copy(), sample_rate=sr)
    axes[i+1].plot(aug_audio, lw=0.5, color=f'C{i+1}')
    axes[i+1].set_title(f'Augmented Version {i+1}')
    axes[i+1].set_ylabel('Amplitude')

plt.suptitle('E1: Audio Augmentation (audiomentations)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# ── E2: Spectrogram augmentation (SpecAugment) ───────────────────────────────
def spec_augment(mel_spec, freq_mask_param=15, time_mask_param=30, n_freq=2, n_time=2):
    """SpecAugment: mask frequency and time bands."""
    spec = mel_spec.copy()
    n_mel, n_steps = spec.shape
    # Frequency masking
    for _ in range(n_freq):
        f  = np.random.randint(0, freq_mask_param)
        f0 = np.random.randint(0, n_mel - f)
        spec[f0:f0+f, :] = spec.mean()
    # Time masking
    for _ in range(n_time):
        t  = np.random.randint(0, time_mask_param)
        t0 = np.random.randint(0, n_steps - t)
        spec[:, t0:t0+t] = spec.mean()
    return spec

mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=64)
mel_db = librosa.power_to_db(mel, ref=np.max)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(mel_db, aspect='auto', origin='lower', cmap='inferno')
axes[0].set_title('Original Mel Spectrogram')
for i in range(1, 4):
    aug_mel = spec_augment(mel_db)
    axes[i].imshow(aug_mel, aspect='auto', origin='lower', cmap='inferno')
    axes[i].set_title(f'SpecAugment v{i}')
for ax in axes: ax.axis('off')
plt.suptitle('E2: SpecAugment (Frequency + Time Masking)', fontsize=12)
plt.tight_layout(); plt.show()

---
## PART F — AugLy (Facebook Research) — Multi-modal

In [ ]:
import augly.image as imaugs
import augly.text  as txtaugs
from PIL import Image

# ── F1: AugLy image augmentation ────────────────────────────────────────────
# Create a simple test image (or use CIFAR sample)
sample_pil = Image.fromarray(x_train_raw[0])

augly_image_augs = {
    'Original'         : lambda img: img,
    'Blur'             : lambda img: imaugs.blur(img, radius=2),
    'Brightness'       : lambda img: imaugs.brightness(img, factor=1.8),
    'Grayscale'        : lambda img: imaugs.grayscale(img),
    'HFlip'            : lambda img: imaugs.hflip(img),
    'Rotate'           : lambda img: imaugs.rotate(img, degrees=20),
    'Pixelize'         : lambda img: imaugs.pixelization(img, ratio=0.3),
    'RandomNoise'      : lambda img: imaugs.random_noise(img, mean=0, var=0.01),
    'Saturation'       : lambda img: imaugs.saturation(img, factor=2.0),
    'Sharpen'          : lambda img: imaugs.sharpen(img, factor=2.0),
    'PerspectiveWarp'  : lambda img: imaugs.perspective_transform(img, sigma=50),
}

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, (name, aug_fn) in zip(axes.flat, augly_image_augs.items()):
    try:
        out = aug_fn(sample_pil.copy())
        if isinstance(out, Image.Image):
            ax.imshow(out)
        else:
            ax.imshow(sample_pil)
    except Exception as e:
        ax.imshow(sample_pil)
        name += ' (err)'
    ax.set_title(name, fontsize=8); ax.axis('off')
plt.suptitle('F1: AugLy Image Augmentation', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── F2: AugLy text augmentation ──────────────────────────────────────────────
print('=== F2: AugLy Text Augmentation ===')
text_samples = [
    "Data augmentation is crucial for robust deep learning.",
    "Neural networks require large datasets to generalise well.",
]

augly_text_augs = {
    'ReplaceSimilarUnicodeChars': txtaugs.replace_similar_unicode_chars,
    'InsertPunctuationChars'    : txtaugs.insert_punctuation_chars,
    'SimulateTypos'             : txtaugs.simulate_typos,
    'SplitWords'                : txtaugs.split_words,
    'SwapGenderedWords'         : txtaugs.swap_gendered_words,
}

for t in text_samples:
    print(f'\nOriginal: {t}')
    for name, aug_fn in augly_text_augs.items():
        try:
            out = aug_fn(t)
            if isinstance(out, list): out = out[0]
            print(f'  {name}: {out}')
        except Exception as e:
            print(f'  {name}: [error: {e}]')

---
## PART G — Document Image Augmentation

In [ ]:
# ── G: Document-specific augmentation with albumentations ───────────────────
# Simulate a document-like image (white background, dark text)
def make_doc_image(h=128, w=128):
    img = np.full((h, w, 3), 240, dtype=np.uint8)  # light gray bg
    # Fake text lines
    for y in range(15, h-10, 12):
        x_start = np.random.randint(5, 20)
        x_end   = np.random.randint(80, w-5)
        thickness = np.random.randint(1, 3)
        cv2.line(img, (x_start, y), (x_end, y), (30, 30, 30), thickness)
    return img

doc_img = make_doc_image()

doc_aug_pipeline = A.Compose([
    A.Rotate(limit=5, border_mode=cv2.BORDER_CONSTANT, value=240, p=0.8),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.7),
    A.GaussNoise(var_limit=(5, 25), p=0.5),
    A.Blur(blur_limit=3, p=0.3),
    A.ElasticTransform(alpha=0.5, sigma=10, p=0.3),
    A.GridDistortion(num_steps=2, distort_limit=0.05, p=0.3),
    A.CoarseDropout(max_holes=5, max_height=8, max_width=30,
                   fill_value=240, p=0.4),   # ink dropout / scanning artefacts
    A.Perspective(scale=(0.02, 0.05), p=0.4),
])

fig, axes = plt.subplots(2, 6, figsize=(18, 5))
axes[0,0].imshow(doc_img); axes[0,0].set_title('Original Doc', fontsize=9)
axes[0,0].axis('off')
for i, ax in enumerate(list(axes.flat)[1:], 1):
    aug = doc_aug_pipeline(image=doc_img)['image']
    ax.imshow(aug, cmap='gray'); ax.set_title(f'Aug #{i}', fontsize=9); ax.axis('off')
plt.suptitle('G: Document Image Augmentation (scanning artefacts, skew, noise)', fontsize=12)
plt.tight_layout(); plt.show()

---
## Summary — All Augmentation Techniques

In [ ]:
summary = {
    'Modality': ['Image','Image','Image','Image','Text','Text','Text',
                 'Time Series','Time Series','Tabular','Tabular','Tabular',
                 'Speech','Speech','Video/Multi','Document'],
    'Technique': [
        'Random Flip/Rotate/Zoom (TF Keras)','Albumentations pipeline',
        'RandAugment','CutMix / MixUp',
        'Synonym Replacement (nlpaug)','Contextual Word Embedding Insert',
        'AugLy text transforms',
        'TimeWarp (tsaug)','AddNoise + Drift',
        'Gaussian Feature Noise','SMOTE','Feature Dropout',
        'audiomentations (pitch/speed/noise)','SpecAugment',
        'AugLy (multi-modal)','Albumentations (skew/blur/dropout)'
    ],
    'Library': [
        'tensorflow.keras','albumentations','keras_cv','keras_cv',
        'nlpaug','nlpaug','augly',
        'tsaug','tsaug',
        'numpy','imbalanced-learn','numpy',
        'audiomentations','librosa+numpy',
        'augly','albumentations'
    ],
    'Use Case': [
        'Classification','Classification','Classification','Classification',
        'Text Classification','NLU','Robustness',
        'Time Series Clf','Regression/Clf',
        'Imbalanced tables','Imbalanced tables','Regularisation',
        'ASR / Audio Clf','ASR',
        'General','Document OCR/Clf'
    ]
}

df_summary = pd.DataFrame(summary)
print('\n=== Multi-modal Augmentation Summary ===')
print(df_summary.to_string(index=False))

# Colour by modality
modality_colors = {
    'Image':'#3498db','Text':'#e74c3c','Time Series':'#2ecc71',
    'Tabular':'#9b59b6','Speech':'#f39c12','Video/Multi':'#1abc9c','Document':'#e67e22'
}

fig, ax = plt.subplots(figsize=(10, len(df_summary)*0.45 + 1))
ax.axis('off')
tbl = ax.table(
    cellText=df_summary.values,
    colLabels=df_summary.columns,
    cellLoc='left', loc='center'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
tbl.auto_set_column_width([0,1,2,3])
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
    elif r > 0 and c == 0:
        mod = df_summary['Modality'].iloc[r-1]
        cell.set_facecolor(modality_colors.get(mod, '#ecf0f1'))
        cell.set_alpha(0.5)
plt.title('Multi-modal Augmentation Summary', fontsize=13, pad=20)
plt.tight_layout(); plt.show()